In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

rna_adata_all = sc.read_h5ad('PapalexiSatija2021_eccite_arrayed_RNA.h5ad')
print(rna_adata_all)

out_path = '../data'

In [ ]:
rna_perturbation = rna_adata_all[rna_adata_all.obs['perturbation'] != 'control', :].obs['perturbation'].astype(str)
perturbs = np.unique(rna_perturbation)
genes = np.unique(rna_adata_all.var_names)
for perturb in perturbs:
    if perturb in genes:
        continue
    else:
        print(perturb+' not in gene_names, remove it')
perturbation_categorical = rna_adata_all.obs['perturbation']
if 'CD274' not in perturbation_categorical.cat.categories:
    perturbation_categorical = perturbation_categorical.cat.add_categories(['CD274'])

rna_adata_all.obs['perturbation'] = perturbation_categorical
rna_adata_all.obs.loc[rna_adata_all.obs['perturbation'] == 'PDL1', 'perturbation'] = 'CD274'
perturbs = ['CD274' if perturb == 'PDL1' else perturb for perturb in perturbs]
print(np.unique(rna_adata_all.obs['perturbation']))

def preprocess_rna(rna_adata_all, perturbs, n_top_genes=5000):
    sc.pp.filter_genes(rna_adata_all, min_cells = 10)
    sc.pp.normalize_total(rna_adata_all, target_sum = 10000)
    sc.pp.log1p(rna_adata_all)
    return rna_adata_all
print(rna_adata_all)
rna_adata_all = preprocess_rna(rna_adata_all, perturbs, 5000)
print(rna_adata_all)

In [ ]:
control_str = 'control'
rna_adata_all.obs['perturbation'] = rna_adata_all.obs['perturbation'].replace(control_str, 'ctrl')
rna_adata_all.obs['perturbation'] = rna_adata_all.obs['perturbation'].apply(
    lambda x: f'{x}+ctrl' if isinstance(x, str) and not x.endswith('ctrl') else x
)
print(np.unique(rna_adata_all.obs['perturbation']))

In [ ]:
rna_adata_new = sc.AnnData(
    X=rna_adata_all.X,
    obs=pd.DataFrame({
        'condition': rna_adata_all.obs['perturbation'],
        'cell_type': rna_adata_all.obs['celltype']
    }),
    var=pd.DataFrame({
        'gene_name': rna_adata_all.var.index
    })
)
rna_adata_new.write(out_path + '/processed_rna.h5ad')

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

adata = sc.read(out_path + '/processed_rna.h5ad')

In [ ]:
import sys
from gears import PertData
pert_data = PertData('../data', default_pert_graph=False)
pert_data.new_data_process(dataset_name = 'papalexisatija2021', adata = adata)
pert_data.load(data_path = '../data/papalexisatija2021')
print(len(pert_data.pert_names))

In [ ]:
import pickle
import numpy as np
custom_split = {'train':['ATF2+ctrl', 'CAV1+ctrl', 'CD274+ctrl', 'ETV7+ctrl', 'IFNGR1+ctrl',
       'IFNGR2+ctrl', 'IRF1+ctrl', 'IRF7+ctrl', 'MARCH8+ctrl',
       'STAT2+ctrl', 'ctrl'],
                'val':['ctrl'],
                'test':['ctrl']}
with open('../data/custom_split.pkl', 'wb') as f:
    pickle.dump(custom_split, f)

pert_data.prepare_split(split = 'custom', split_dict_path='../data/custom_split.pkl', seed = 1)
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

In [ ]:
from gears import PertData, GEARS
gears_model = GEARS(pert_data, device = 'cuda:0', 
                        weight_bias_track = False, 
                        proj_name = 'pertnet', 
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64)
gears_model.train(epochs = 20, lr = 1e-3)

pert_embeddings, pert_list = gears_model.get_pert_embeddings()
temp =[]
for gene in gears_model.gene_list:
    if gene not in gears_model.pert_list:
        temp.append(gene)
print(str(len(temp))+' genes cannot embed by GEARS')
np.savez('../data/pert_embeddings.npz', pert_embeddings=pert_embeddings, pert_list=pert_list)